# 01 - Data Ingestion

**Objective:** Load raw Seoul bike sharing data, clean it, and save processed output.

**Dataset:** Seoul Bike Sharing Demand (UCI)

**Steps:**
1. Load raw data from CSV
2. Clean and one-hot encode categorical features
3. Check data quality (missing values, dtypes)
4. Save processed data


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully")


### Dataset Attributes

Understanding each column helps you make better decisions during cleaning and feature engineering.

- **Date:** Date of record — not useful for regression modeling since it's just a timestamp
- **Rented Bike Count:** Target variable (continuous, what we predict)
- **Hour:** Hour of the day (0-23)
- **Temperature(C):** Temperature in Celsius
- **Humidity(%):** Humidity percentage
- **Wind speed (m/s):** Wind speed
- **Visibility (10m):** Visibility in 10m units
- **Dew point temperature(C):** Dew point temperature
- **Solar Radiation (MJ/m2):** Solar radiation
- **Rainfall(mm):** Rainfall in mm
- **Snowfall (cm):** Snowfall in cm
- **Seasons:** Winter, Spring, Summer, Autumn — categorical
- **Holiday:** Yes/No — categorical
- **Functioning Day:** Yes/No — categorical


In [ ]:
# TODO: Load the raw CSV data into a DataFrame
# The Seoul bike sharing dataset uses unicode encoding, so pass encoding="unicode_escape"
# to pd.read_csv() so that special characters (like degree symbols in column names) are read correctly.

RAW_DIR = Path("../data/raw")

# TODO: After loading, inspect the data to make sure it looks right
# Use .shape to check how many rows and columns we have,
# and .head() to preview the first few rows. It's also a good idea to
# check the data types with .info() so you know which columns are numeric vs object.

# print(f"Shape: {df.shape}")
# print(df.head())
# df.info()


In [ ]:
# TODO Ensure that the "Date" column is in datetime format. If it's not, convert it using pd.to_datetime().
# df["Date"] = pd.to_datetime(df["Date"])

# TODO: Use .describe() to get summary statistics for the numeric columns. This will help you understand the distribution of the data and identify any potential outliers.
# print(df.describe())


In [ ]:
# TODO: Use .value_counts() on a categorical column (like "Season" or "Holiday") to see the distribution of categories. This can help you understand if the dataset is balanced or if there are any dominant categories.
# print(df["Season"].value_counts())

In [ ]:
# TODO: Check for missing values in each column
# Use df.isnull().sum() to get a count of NaN entries per column.
# This tells you which columns have gaps and how many rows need attention.

# TODO: Handle missing values if any are found
# Two common strategies:
#   - df.dropna() — removes any row with a missing value (simple, but you lose data)
#   - df.fillna(value) — replaces missing entries with something like the column mean or median
# Choose the approach that makes sense for your data.

# TODO: Reset the DataFrame index after dropping rows (if you used dropna)
# df.reset_index(drop=True, inplace=True)



In [ ]:
# TODO: Drop any irrelevant columns that won't be used for modeling (if any)
# For example, if there are columns that are just identifiers or have too many unique values, they might not be useful for prediction and could be dropped to simplify the dataset.
# df.drop(columns=["Irrelevant_Column"], inplace=True) # 

In [ ]:
# TODO: Change column names to be more Pythonic (optional but recommended) - data specific. 
# For example, if you have a column named "Temperature(°C)", you might want to rename it to "temperature_c" for easier access in code.
# df.rename(columns={"Temperature(°C)": "temperature_c"}, inplace=True)

In [ ]:
# TODO: Find duplicate rows in the DataFrame
# Use df.duplicated() to identify duplicate rows. 
# This returns a boolean Series where True indicates a duplicate row (except for the first occurrence).
# You can sum this Series to get the total count of duplicates.


# TODO: Duplicates can be searched for across all columns or a subset of columns.
# e.g. df.duplicated(subset=["Date", "Hour"])

# TODO: Remove duplicate rows if any are found
# Use df.drop_duplicates() to remove duplicate rows.
# param: keep="first": it keeps the first occurrence and drops subsequent duplicates. alternatives are "last" (keeps the last occurrence) or False (drops all duplicates).
# You can specify subset=["Date", "Hour"] to drop duplicates based on just those columns.

# TODO: Alternatively, you can use df.duplicated() to filter the DataFrame and inspect duplicates before dropping them.

# TODO: Reset the index after dropping duplicates

In [ ]:
# TODO: Verify the final data is clean
# print(f"Final shape: {df.shape}")
# print(df.head())


# TODO: Save the processed data to data/processed/clean_data.csv
# Use df.to_csv() with index=False so we don't write the DataFrame index as a column.

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# df.to_csv(PROCESSED_DIR / "clean_data.csv", index=False)
# print("Clean data saved successfully")


In [ ]:
# TODO: Verify the saved file by loading it back into a new DataFrame
# This step confirms the file wasn't corrupted during the write process.
# Compare the shape of the loaded data to the original shape.

# df_check = pd.read_csv(PROCESSED_DIR / "clean_data.csv")
# print(f"Loaded data shape: {df_check.shape}")
# assert df_check.shape == df.shape, "Shape mismatch after save-load cycle"
# print("Verification passed: data round-trips correctly")


## 2. Relational Data — Joining Tables

In real-world projects, data is rarely handed to you in a single tidy CSV. 
It is often spread across multiple tables in a database — a **fact** table (the core measurements)
and several **dimension** tables (contextual information), linked by key columns.

To build your feature matrix you need to **join** these tables back together.

In this section you will work with a relational version of the Seoul bike data,
split into three CSV files that simulate a real database:

| File | Content | Key column |
|------|---------|------------|
| `bike_rentals.csv` | Core rental transactions — date, hour, count | `rental_id` |
| `weather_conditions.csv` | Weather sensor readings | `weather_id` |
| `time_context.csv` | Calendar and context info | `context_id` |

Some IDs appear in only some tables — exactly like a real database where sensors 
go offline or records are orphaned. Your job is to re-assemble the full dataset.

In [ ]:
# Load the three relational tables
RELATIONAL_DIR = Path("../data/raw/relational")

rentals = pd.read_csv(RELATIONAL_DIR / "bike_rentals.csv")
weather = pd.read_csv(RELATIONAL_DIR / "weather_conditions.csv")
context = pd.read_csv(RELATIONAL_DIR / "time_context.csv")

# TODO: Inspect each table
# Check the shape and first few rows of each DataFrame.
# How many IDs does each table have? Are the ID ranges the same?

# print(f"Rentals: {rentals.shape}")
# print(f"Weather: {weather.shape}")
# print(f"Context: {context.shape}")
# print(rentals.head())
# print(weather.head())
# print(context.head())


In [ ]:
# TODO: Identify which rental records are missing weather data
# Use pd.merge() to join rentals with weather on rental_id == weather_id.
# Start with a LEFT JOIN so you keep all rentals. Then check how many
# weather columns are NaN — those are the rentals without matching weather records.

# rentals_weather = pd.merge(rentals, weather, left_on="rental_id", right_on="weather_id", how="left")

# TODO: Now try an INNER JOIN instead. How many rows do you lose? Why?


In [ ]:
# TODO: Merge in the time_context table too
# Chain a second merge to bring in calendar context.
# rentals_weather joins on rental_id, then merge that result with context
# using left_on="rental_id" and right_on="context_id".

# TODO: Check how many rows have missing context (Seasons == NaN)


In [ ]:
# TODO: Compare the merged DataFrame to the original single-table shape
# The original SeoulBikeData.csv has 8760 rows × 14 columns.
# After splitting and re-joining, do we get the same number of rows?
# Fewer? More? What does this tell you about the completeness of each table?

# HINT: If some rental IDs have no matching weather or context,
# a LEFT JOIN keeps the rental row with NaN fill. An INNER JOIN drops it.
# How many rows would an INNER JOIN of all three tables produce?


### Exercises

1. **Explore encoding options**: Try using `drop_first=False` in `pd.get_dummies()` and observe how the column count changes.
2. **Experiment with missing value handling**: Intentionally introduce missing values with `df.loc[..., ...] = np.nan`, then try both `dropna()` and `fillna()` and compare the resulting shapes.
3. **Save in Parquet format**: Use `df.to_parquet()` instead of CSV and compare file sizes. Parquet is often faster and more space-efficient.
4. **Add a new feature**: Create a "weekend" flag by checking if the date falls on a Saturday or Sunday using `pd.to_datetime()`.
5. **What would break?**: If the raw CSV file format changed (new columns, different encoding), which part of this notebook would fail first?
6. **Which JOIN?**: You used LEFT JOINs to assemble the relational tables. Try replacing them with INNER JOINs. How many rows do you lose? Why? What does that imply about data completeness in each table?
7. **Fake data detective**: The `weather_conditions.csv` and `time_context.csv` each contain a few artificially generated rows (IDs > 8760). Can you identify them? What heuristic did you use — missing rental matches, unusual value patterns, or something else?
8. **Date as a join key**: Both `rental_id` and `Date` appear in all three tables. Try merging on `Date` instead of ID. Does the row count change? Why might date-only joins be dangerous in a real database (think: duplicate dates across hours)?
